# STRAW Residual + Hypernetwork Exploration

Notebook goal:
- Inspect residual/attention-output tensor shapes from Mistral
- Prototype small BERT hypernetwork that predicts LoRA A/B for `v_proj`
- Validate shape compatibility before full training integration

In [9]:
! pip install torch -q
! pip install torchinfo -q
! pip install transformers -q

In [10]:
import sys
from pathlib import Path

import torch
from torchinfo import summary
from transformers import AutoModelForCausalLM, AutoTokenizer

# Make project imports work whether notebook is launched from repo root or notebooks/
cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent]
for base in candidates:
    if (base / "experiments").exists():
        sys.path.insert(0, str(base))
        break

from experiments.straw.hooks import register_attention_output_hooks, clear_hooks
from experiments.straw.hypernet_bert import BertHyperLoraGenerator

In [11]:
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=DTYPE,
)
model = model.to(DEVICE).eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

Loading weights: 100%|██████████| 291/291 [00:00<00:00, 1126.62it/s, Materializing param=model.norm.weight]                              


In [12]:
messages = [
    {"role": "system", "content": "You are a careful assistant."},
    {"role": "user", "content": "2+2? A.3 B.4 C.5 D.6 Return only label."},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}

In [13]:
capture, handles = register_attention_output_hooks(model)
with torch.no_grad():
    _ = model(**inputs)
clear_hooks(handles)

num_layers = len(capture.per_layer)
sample_layer = 0
sample_tensor = capture.per_layer[sample_layer]
print("captured_layers:", num_layers)
print("sample_layer_shape:", tuple(sample_tensor.shape))

captured_layers: 32
sample_layer_shape: (1, 32, 4096)


In [20]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE).to(DEVICE).eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(model.__class__.__name__)
print("hidden_size:", model.config.hidden_size)
print("num_layers:", model.config.num_hidden_layers)
print("num_heads:", model.config.num_attention_heads)
print("num_kv_heads:", model.config.num_key_value_heads)

Loading weights: 100%|██████████| 291/291 [00:00<00:00, 1106.36it/s, Materializing param=model.norm.weight]                              


MistralForCausalLM
hidden_size: 4096
num_layers: 32
num_heads: 32
num_kv_heads: 8


In [21]:
# High-level tree
print(model.model)  # decoder stack, norm
print(model.lm_head)

# First layer attention projections
l0 = model.model.layers[0].self_attn
print("q_proj:", l0.q_proj.in_features, "->", l0.q_proj.out_features)
print("k_proj:", l0.k_proj.in_features, "->", l0.k_proj.out_features)
print("v_proj:", l0.v_proj.in_features, "->", l0.v_proj.out_features)
print("o_proj:", l0.o_proj.in_features, "->", l0.o_proj.out_features)

MistralModel(
  (embed_tokens): Embedding(32768, 4096)
  (layers): ModuleList(
    (0-31): 32 x MistralDecoderLayer(
      (self_attn): MistralAttention(
        (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
        (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
        (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
        (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
      )
      (mlp): MistralMLP(
        (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
        (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
        (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
      (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
    )
  )
  (norm): MistralRMSNorm((4096,), eps=1e-05)
  (rotary_emb): MistralRotaryEmbedding()
)
Linear(in_featu

In [22]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is 2+2?"},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

with torch.no_grad():
    out = model(**inputs, output_hidden_states=True, use_cache=False)

hidden_states = out.hidden_states  # tuple length = num_layers + 1
print("num_hidden_states:", len(hidden_states))  # embeddings + each layer output

for i, h in enumerate(hidden_states[:5]):
    print(f"hidden_states[{i}] shape:", tuple(h.shape))
print("...")
print(f"hidden_states[-1] shape: {tuple(hidden_states[-1].shape)}")

num_hidden_states: 33
hidden_states[0] shape: (1, 19, 4096)
hidden_states[1] shape: (1, 19, 4096)
hidden_states[2] shape: (1, 19, 4096)
hidden_states[3] shape: (1, 19, 4096)
hidden_states[4] shape: (1, 19, 4096)
...
hidden_states[-1] shape: (1, 19, 4096)


In [24]:
layer_shapes = {}

handles = []
for i, layer in enumerate(model.model.layers):
    def make_hook(idx):
        def hook(_m, _inp, out):
            t = out[0] if isinstance(out, tuple) else out
            layer_shapes[idx] = tuple(t.shape)
        return hook
    handles.append(layer.register_forward_hook(make_hook(i)))

with torch.no_grad():
    _ = model(**inputs, use_cache=False)

for h in handles:
    h.remove()

print("Captured layer outputs:")
for i in range(min(5, len(layer_shapes))):
    print(f"layer {i} -> {layer_shapes[i]}")
print("...")
print(f"layer {model.config.num_hidden_layers-1} -> {layer_shapes[model.config.num_hidden_layers-1]}")

Captured layer outputs:
layer 0 -> (1, 19, 4096)
layer 1 -> (1, 19, 4096)
layer 2 -> (1, 19, 4096)
layer 3 -> (1, 19, 4096)
layer 4 -> (1, 19, 4096)
...
layer 31 -> (1, 19, 4096)


In [17]:
# Build a compact hypernetwork prototype (LoRA-contract aligned)
hidden_size = model.config.hidden_size
rank = 8

v_proj = model.model.layers[0].self_attn.v_proj
v_proj_in = v_proj.in_features
v_proj_out = v_proj.out_features

# Prefix-conditioned input: pool first few prompt tokens only
prefix_len = min(sample_tensor.shape[1], 8)
prefix_hidden = sample_tensor[:, :prefix_len, :].mean(dim=1)

hypernet = BertHyperLoraGenerator(
    residual_dim=hidden_size,
    num_layers=model.config.num_hidden_layers,
    v_proj_in=v_proj_in,
    v_proj_out=v_proj_out,
    rank=rank,
).to(model.device)

# Keep hypernet in float32 for stability; cast input to match
hypernet_dtype = next(hypernet.parameters()).dtype
prefix_hidden = prefix_hidden.to(device=model.device, dtype=hypernet_dtype)

# torchinfo may fail on dataclass outputs; keep as best-effort only
try:
    summary(hypernet, input_data=prefix_hidden)
except Exception as e:
    print("torchinfo skipped (non-tensor structured output):", type(e).__name__, e)
    total_params = sum(p.numel() for p in hypernet.parameters())
    trainable_params = sum(p.numel() for p in hypernet.parameters() if p.requires_grad)
    print(f"hypernet params: total={total_params:,}, trainable={trainable_params:,}")

with torch.no_grad():
    out = hypernet(prefix_hidden)
print("A shape:", tuple(out.a.shape))
print("B shape:", tuple(out.b.shape))

torchinfo skipped (non-tensor structured output): RuntimeError Failed to run torchinfo. See above stack traces for more details. Executed layers up to: [Linear: 1, BertModel: 1, BertEmbeddings: 2, Embedding: 3, Embedding: 3, LayerNorm: 3, Dropout: 3, BertEncoder: 2, BertLayer: 4, BertAttention: 5, BertSelfAttention: 6, Linear: 7, Linear: 7, Linear: 7, BertSelfOutput: 6, Linear: 7, Dropout: 7, LayerNorm: 7, BertIntermediate: 5, Linear: 6, GELUActivation: 6, BertOutput: 5, Linear: 6, Dropout: 6, LayerNorm: 6, BertLayer: 4, BertAttention: 5, BertSelfAttention: 6, Linear: 7, Linear: 7, Linear: 7, BertSelfOutput: 6, Linear: 7, Dropout: 7, LayerNorm: 7, BertIntermediate: 5, Linear: 6, GELUActivation: 6, BertOutput: 5, Linear: 6, Dropout: 6, LayerNorm: 6, BertLayer: 4, BertAttention: 5, BertSelfAttention: 6, Linear: 7, Linear: 7, Linear: 7, BertSelfOutput: 6, Linear: 7, Dropout: 7, LayerNorm: 7, BertIntermediate: 5, Linear: 6, GELUActivation: 6, BertOutput: 5, Linear: 6, Dropout: 6, LayerNorm

In [ ]:
print(model)

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32768, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): MistralRMSNorm((4096,)

: 

Next:
1. Choose Config B (rank, hypernet hidden size/layers)
2. Add v_proj injection utility with these A/B tensors
3. Move from shape-validation to train loop integration